# Solution A1 · A Pipeline from scratch: who calls `fit` and who calls `transform`

**Type**: Implement from scratch

## Objective

Implement a `MyPipeline` class with `fit`, `predict`, `predict_proba`, `named_steps` and `set_params`, check it against
`sklearn.pipeline.Pipeline` and show, by counting the calls, that in `predict` no intermediate step is refitted.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

df = ds.titanic()
NUM = ["pclass", "age", "siblings_spouses", "parents_children", "fare"]
X_tr, X_te, y_tr, y_te = train_test_split(df[NUM], df["survived"], test_size=0.25, stratify=df["survived"], random_state=0)


class Spy:
    """Wraps a transformer and counts the calls to fit_transform and transform."""

    def __init__(self, t):
        self.t, self.n_fit_transform, self.n_transform = t, 0, 0

    def fit_transform(self, X, y=None):
        self.n_fit_transform += 1
        return self.t.fit_transform(X, y)

    def transform(self, X):
        self.n_transform += 1
        return self.t.transform(X)

    def fit(self, X, y=None):                      # a correct pipeline must never use it
        raise RuntimeError("the pipeline must not call fit on the intermediate steps")

## Commented solution

In [ ]:
class MyPipeline:
    def __init__(self, steps):
        self.steps = steps
        self.named_steps = dict(steps)

    def _transform(self, X):
        for _, step in self.steps[:-1]:
            X = step.transform(X)                       # in prediction: ONLY transform
        return X

    def fit(self, X, y):
        for _, step in self.steps[:-1]:
            X = step.fit_transform(X, y)                # in training: fit_transform on the intermediate steps
        self.steps[-1][1].fit(X, y)
        return self

    def predict(self, X):
        return self.steps[-1][1].predict(self._transform(X))

    def predict_proba(self, X):
        return self.steps[-1][1].predict_proba(self._transform(X))

    def set_params(self, **kw):
        for key, value in kw.items():
            name, parameter = key.split("__", 1)
            self.named_steps[name].set_params(**{parameter: value})
        return self

In [ ]:
spy = Spy(StandardScaler())
mine = MyPipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", spy),
                   ("model", LogisticRegression(max_iter=1000))])
mine.fit(X_tr, y_tr)
for _ in range(3):
    p = mine.predict_proba(X_te)
spy_fit_transform = spy.n_fit_transform
spy_transform = spy.n_transform
print("fit_transform:", spy_fit_transform, "| transform:", spy_transform)

## Check

In [ ]:
with verify():
    ref = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),
                    ("model", LogisticRegression(max_iter=1000))]).fit(X_tr, y_tr)
    check(close(mine.predict_proba(X_te), ref.predict_proba(X_te)), "the probabilities match those of sklearn.Pipeline",
          "MyPipeline.predict_proba must give the same result as Pipeline")
    check(close(mine.predict(X_te), ref.predict(X_te)), "the predicted classes match", "predict must use the same transformations")
    check(list(mine.named_steps) == ["imputer", "scaler", "model"], "named_steps is a dictionary name -> object", "named_steps = dict(steps)")
    check(spy_fit_transform == 1, "in training the scaler did fit_transform just once", "fit_transform must be called once, in fit")
    check(spy_transform == 3, "in the three predictions the scaler did only transform (3 times)", "in predict only transform is called: once per prediction")
    mine.set_params(model__C=0.01)
    check(mine.named_steps["model"].C == 0.01, "set_params reaches the parameters of the last step", "set_params('model__C') must modify the model")
    mine.fit(X_tr, y_tr)
    check(not np.allclose(mine.predict_proba(X_te), ref.predict_proba(X_te)), "after set_params and a new fit the model changes", "small C = strong regularization: the probabilities change")
done()

## What you learned

- The rule is just one: **`fit_transform` in training, `transform` in prediction**. It is enough to guarantee that no statistic
  of the test set enters the preprocessing, because in prediction nothing gets computed.
- `GridSearchCV` and `cross_val_score` use the same rule: they clone the pipeline and refit it on the training part of every fold alone. Preprocessing
  leakage becomes impossible by construction.
- `set_params` with the `step__parameter` syntax is what makes the hyperparameter search inside the pipeline possible (lesson 3.3).

## Common pitfall

Calling `fit` (instead of `fit_transform`) on the intermediate steps and then `transform` on the same data is equivalent. Calling
`fit_transform` in **prediction** is not: it recomputes the statistics on the batch it receives, the bug of exercise E3 of module 2.

## Interview question

> *Click on a question to show its answer.*

<details><summary>What happens, step by step, when you call `cross_val_score(pipeline, X, y, cv=5)`?</summary>

At every fold `cross_val_score` clones the pipeline (parameters, not state), trains it on the training part of the fold (`fit`: the intermediate
steps do `fit_transform`, the last one `fit`) and evaluates it on the validation fold (`predict`: only `transform` and then
`predict`). The preprocessing statistics are therefore recomputed 5 times, each only on the training data of the fold: it is the
reason why there is no leakage.

</details>